# 02 Data Cleaning

This notebook shows how the cleaned modeling table is produced through the validated preprocessing module.

## Reproducibility note

These notebooks are lightweight narrative companions to the source-controlled pipeline. The canonical workflow lives in src, generated metrics and figures live under reports, and the final selected model is models/xgboost_public.pkl. The protected attribute SEX is excluded from active model training and retained only for fairness diagnostics.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, Markdown

from src.utils import ROOT_DIR, REPORTS_DIR, MODELS_DIR, load_dataset_auto, load_model
from src.data_preprocessing import TARGET_COL

ROOT = ROOT_DIR
REPORTS = REPORTS_DIR
MODELS = MODELS_DIR
PROCESSED_DATA = ROOT / 'data' / 'processed' / 'uci_taiwan_credit_default_processed.csv'

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')


def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)


def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing optional image: {path.relative_to(ROOT) if path.is_absolute() else path}')


def load_project_dataset():
    try:
        frame, source = load_dataset_auto()
        return frame, source
    except Exception as exc:
        if PROCESSED_DATA.exists():
            print(f'UCI loader failed; using processed local fallback. Reason: {exc}')
            return pd.read_csv(PROCESSED_DATA), PROCESSED_DATA
        raise

raw_df, DATA_SOURCE = load_project_dataset()
print('Project root:', ROOT)
print('Data source:', DATA_SOURCE)
print('Rows:', f'{len(raw_df):,}')
print('Columns:', raw_df.shape[1])

from src.data_preprocessing import prepare_modeling_table
from src.utils import normalize_target

clean_df = normalize_target(raw_df, target_col=TARGET_COL)
prepared = prepare_modeling_table(raw_df, target_col=TARGET_COL)
print('Raw shape:', raw_df.shape)
print('Prepared shape:', prepared.shape)
print('Target missing after normalization:', int(clean_df[TARGET_COL].isna().sum()))

Project root: D:\PGDBA\Projects\Credit Default Risk\credit-default-xai
Data source: uci:\default_credit_card
Rows: 30,000
Columns: 38


Raw shape: (30000, 38)
Prepared shape: (30000, 38)
Target missing after normalization: 0


## Cleaning choices

- Standardize the target as a binary integer Default_Flag.
- Drop ID-style columns when present.
- Add validated ratio and repayment-history features.
- Keep protected attributes available for audit tables, not active model training.

In [2]:
comparison = pd.DataFrame({
    'raw_columns': pd.Series(raw_df.columns),
    'prepared_columns': pd.Series(prepared.columns),
})
display(comparison.head(40))
display(prepared[[TARGET_COL]].value_counts().rename('rows').reset_index())

,raw_columns,prepared_columns
0,LIMIT_BAL,LIMIT_BAL
1,SEX,SEX
2,EDUCATION,EDUCATION
3,MARRIAGE,MARRIAGE
4,AGE,AGE
5,PAY_0,PAY_0
6,PAY_2,PAY_2
7,PAY_3,PAY_3
8,PAY_4,PAY_4
9,PAY_5,PAY_5


,Default_Flag,rows
0,0,23364
1,1,6636
